# LeWM Training
Run cells top to bottom. Checkpoints save to Google Drive after each epoch.

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
import subprocess, os, sys, re, threading, shutil, glob, time
subprocess.run(['pip', 'install', '-q', 'stable-worldmodel[train]'], check=True)
print('Done')

In [ ]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
DRIVE_CKPT = '/content/drive/MyDrive/lewm_checkpoints'
os.makedirs(DRIVE_CKPT, exist_ok=True)
print(f'Checkpoints will be saved to: {DRIVE_CKPT}')

In [ ]:
# ── Clone repo + patch config ─────────────────────────────────────────────────
if not os.path.exists('/content/le-wm'):
    subprocess.run(['git', 'clone', 'https://github.com/UnnatPar/lewm-jamba.git', '/content/le-wm'], check=True)

config_path = '/content/le-wm/config/train/data/pusht.yaml'
with open(config_path) as f:
    cfg = f.read()
cfg = re.sub(r'pusht_expert_train(?:\.lance)?(?:\.h5)?', 'pusht_expert_train.lance', cfg)
with open(config_path, 'w') as f:
    f.write(cfg)
print('Repo ready')

In [ ]:
# ── Download dataset (~12 GB, ~10 min) ───────────────────────────────────────
h5_path  = '/content/stable-wm/datasets/pusht_expert_train.h5'
zst_path = h5_path + '.zst'
os.makedirs('/content/stable-wm/datasets', exist_ok=True)

if not os.path.exists(h5_path):
    dl = subprocess.Popen(
        ['wget', '--progress=dot:giga', '-O', zst_path,
         'https://huggingface.co/datasets/quentinll/lewm-pusht/resolve/main/pusht_expert_train.h5.zst'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    for line in iter(dl.stdout.readline, b''):
        sys.stdout.write(line.decode(errors='replace')); sys.stdout.flush()
    dl.wait()
    if dl.returncode != 0:
        raise SystemExit('wget failed')

    subprocess.run(['apt-get', 'install', '-y', '-q', 'zstd'], check=True)
    print('Decompressing (~5 min)...')
    subprocess.run(['zstd', '-d', zst_path, '-o', h5_path, '--rm'], check=True)

size_gb = os.path.getsize(h5_path) / 1e9
print(f'H5 ready: {size_gb:.1f} GB')
if size_gb < 30:
    raise SystemExit(f'H5 too small — download incomplete')

In [ ]:
# ── Convert H5 → Lance (~20 min, one-time) ────────────────────────────────────
lance_path = '/content/stable-wm/datasets/pusht_expert_train.lance'

lance_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, files in os.walk(lance_path) for f in files
) if os.path.exists(lance_path) else 0

if lance_size < 1_000_000_000:
    if os.path.exists(lance_path):
        shutil.rmtree(lance_path)

    subprocess.run(['pip', 'install', '-q', 'hdf5plugin'], check=True)
    import hdf5plugin, h5py
    from tqdm.notebook import tqdm
    from stable_worldmodel.data.format import get_format

    with h5py.File(h5_path, 'r') as f:
        ep_offsets = f['ep_offset'][:]
        ep_lens    = f['ep_len'][:]
        n_eps      = len(ep_lens)
    print(f'{n_eps} episodes')

    def episodes():
        with h5py.File(h5_path, 'r') as hf:
            for i in tqdm(range(n_eps), desc='converting'):
                o, l = int(ep_offsets[i]), int(ep_lens[i])
                yield {k: list(hf[k][o:o+l]) for k in ['pixels','action','proprio','state']}

    with get_format('lance').open_writer(lance_path, mode='overwrite') as w:
        w.write_episodes(episodes())
    print('Lance ready')
else:
    print(f'Lance already complete ({lance_size/1e9:.1f} GB)')

In [ ]:
# ── Train 10 epochs (checkpoints auto-save to Drive after each epoch) ─────────
CKPT_OUT = '/content/stable-wm/checkpoints'
os.makedirs(CKPT_OUT, exist_ok=True)
_stop = threading.Event()
_seen = set()

def watcher():
    while not _stop.is_set():
        for path in glob.glob('/root/.cache/stable-pretraining/runs/*/*/*/checkpoints/epoch=*.ckpt'):
            if path in _seen:
                continue
            m = re.search(r'epoch=(\d+)', path)
            if not m:
                continue
            try:
                s1 = os.path.getsize(path); time.sleep(5); s2 = os.path.getsize(path)
                if s1 != s2 or s1 == 0:
                    continue
            except OSError:
                continue
            epoch = m.group(1)
            fname = f'weights_epoch_{epoch}.ckpt'
            # copy to local predictable path
            local = os.path.join(CKPT_OUT, fname)
            shutil.copy2(path, local)
            # copy to Drive
            drive_dst = os.path.join(DRIVE_CKPT, fname)
            shutil.copy2(path, drive_dst)
            _seen.add(path)
            size_mb = os.path.getsize(local) / 1e6
            print(f'=== CHECKPOINT SAVED: {fname} ({size_mb:.0f} MB) → Drive + {CKPT_OUT} ===', flush=True)

        _stop.wait(30)

threading.Thread(target=watcher, daemon=True).start()

env = os.environ.copy()
env['LOCAL_DATASET_DIR'] = '/content/stable-wm'
env['STABLEWM_HOME']     = '/content/stable-wm'
env['PYTHONUNBUFFERED']  = '1'

proc = subprocess.Popen(
    ['python', '-u', 'train.py',
     'subdir=lewm_pusht',
     'loader.batch_size=128',
     'num_workers=4',
     '+loader.multiprocessing_context=spawn',
     'trainer.max_epochs=10'],
    cwd='/content/le-wm', env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
for line in iter(proc.stdout.readline, b''):
    sys.stdout.write(line.decode(errors='replace')); sys.stdout.flush()
proc.wait()
_stop.set()

if proc.returncode != 0:
    raise SystemExit(f'train.py failed with code {proc.returncode}')
print('=== TRAINING COMPLETE ===')